#SciBert(Section token)

In [ ]:
# ===== Install required libraries =====
!pip install -q transformers datasets seqeval accelerate

In [ ]:
import os
import json
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [ ]:
from datasets import load_dataset, DatasetDict

# Load each split separately (because schemas differ)
train_ds = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/NLP/train1.json",
    split="train"
)

val_ds = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/NLP/validation.json",
    split="train"
)

test_ds = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/NLP/test.json",
    split="train"
)

# Normalize columns to a single schema
def normalize_columns(example):
    return {
        "title": example.get("title", ""),
        "abstract": example.get("abstract", ""),
        "keywords": (
            example["keywords"]
            if "keywords" in example
            else "; ".join(example["keyphrases"])
            if "keyphrases" in example
            else ""
        )
    }

train_ds = train_ds.map(normalize_columns, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(normalize_columns, remove_columns=val_ds.column_names)
test_ds  = test_ds.map(normalize_columns, remove_columns=test_ds.column_names)

# Rebuild unified dataset
dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})


In [ ]:
def build_section_text(example):
    title = example["title"].strip()
    abstract = example["abstract"].strip()
    text = f"[TITLE] {title} [ABSTRACT] {abstract}"
    return text

In [ ]:
label2id = {"O": 0, "B-KEY": 1, "I-KEY": 2}
id2label = {v: k for k, v in label2id.items()}

In [ ]:
def create_bio_labels(text, keywords):
    tokens = text.split()
    labels = ["O"] * len(tokens)

    keyphrases = [kp.strip().lower() for kp in keywords.split(";")]

    lowered_tokens = [t.lower() for t in tokens]

    for kp in keyphrases:
        kp_tokens = kp.split()
        for i in range(len(tokens) - len(kp_tokens) + 1):
            if lowered_tokens[i:i+len(kp_tokens)] == kp_tokens:
                labels[i] = "B-KEY"
                for j in range(1, len(kp_tokens)):
                    labels[i+j] = "I-KEY"

    return tokens, labels

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "allenai/scibert_scivocab_uncased"
)

In [ ]:
def create_bio_labels(text, keywords):
    tokens = text.split()
    labels = ["O"] * len(tokens)

    # Handle both list and string keyphrases
    if isinstance(keywords, list):
        keyphrases = [kp.lower().strip() for kp in keywords if kp.strip()]
    else:
        keyphrases = [kp.lower().strip() for kp in keywords.split(";") if kp.strip()]

    lowered_tokens = [t.lower() for t in tokens]
    n_tokens = len(tokens)

    for kp in keyphrases:
        kp_tokens = kp.split()
        kp_len = len(kp_tokens)

        # Skip impossible matches
        if kp_len == 0 or kp_len > n_tokens:
            continue

        for i in range(n_tokens - kp_len + 1):
            # Exact token-level match
            if lowered_tokens[i:i + kp_len] == kp_tokens:

                # ✅ SAFE assignment
                labels[i] = "B-KEY"

                for j in range(1, kp_len):
                    if i + j < n_tokens:
                        labels[i + j] = "I-KEY"

    return tokens, labels

In [ ]:
def tokenize_and_align_labels(example):
    text = build_section_text(example)

    words, word_labels = create_bio_labels(
        text,
        example["keywords"]
    )

    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    labels = []
    word_ids = tokenized.word_ids()

    previous_word_id = None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        elif word_id != previous_word_id:
            labels.append(label2id[word_labels[word_id]])
        else:
            # Subword token
            if word_labels[word_id] == "B-KEY":
                labels.append(label2id["I-KEY"])
            else:
                labels.append(label2id[word_labels[word_id]])
        previous_word_id = word_id

    tokenized["labels"] = labels
    return tokenized

In [ ]:
encoded_train = dataset["train"].map(
    tokenize_and_align_labels,
    remove_columns=dataset["train"].column_names
)

encoded_val = dataset["validation"].map(
    tokenize_and_align_labels,
    remove_columns=dataset["validation"].column_names
)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "allenai/scibert_scivocab_uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./scibert-keyphrase",
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=2000,
    eval_steps=2000,
    logging_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    report_to="none"
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("./final_scibert_keyphrase")
tokenizer.save_pretrained("./final_scibert_keyphrase")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("./final_scibert_keyphrase")
model = AutoModelForTokenClassification.from_pretrained(
    "./final_scibert_keyphrase"
).to(device)

model.eval()

In [ ]:
def extract_keyphrases(title, abstract):
    text = f"[TITLE] {title} [ABSTRACT] {abstract}"

    tokens = tokenizer(
        text.split(),
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    tokens = {k: v.to(device) for k, v in tokens.items()}

    with torch.no_grad():
        outputs = model(**tokens)

    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()
    word_ids = tokens["input_ids"].shape

    words = text.split()
    keyphrases = []
    current_phrase = []

    for word, pred in zip(words, predictions):
        label = id2label[pred]
        if label == "B-KEY":
            if current_phrase:
                keyphrases.append(" ".join(current_phrase))
            current_phrase = [word]
        elif label == "I-KEY":
            current_phrase.append(word)
        else:
            if current_phrase:
                keyphrases.append(" ".join(current_phrase))
                current_phrase = []

    if current_phrase:
        keyphrases.append(" ".join(current_phrase))

    return list(set(keyphrases))

In [ ]:
# Check
title = "Federated Learning for Privacy Preserving Medical AI"
abstract = """
This proposal presents a decentralized learning framework where
models are trained locally on hospital data without sharing raw
patient records, ensuring privacy and secure aggregation.
"""

print(extract_keyphrases(title, abstract))

# Metrics


In [ ]:
!pip install -q seqeval scikit-learn pandas

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

In [ ]:
def get_predictions(trainer, dataset):
    predictions, labels, _ = trainer.predict(dataset)

    preds = np.argmax(predictions, axis=2)

    true_labels = []
    true_preds = []

    for label_seq, pred_seq in zip(labels, preds):
        curr_labels = []
        curr_preds = []

        for l, p in zip(label_seq, pred_seq):
            if l != -100:
                curr_labels.append(id2label[l])
                curr_preds.append(id2label[p])

        true_labels.append(curr_labels)
        true_preds.append(curr_preds)

    return true_labels, true_preds

In [ ]:
true_labels, pred_labels = get_predictions(trainer, encoded_val)

precision = precision_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)

print("SciBERT (Section-Aware) Evaluation")
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# TF-IDF(Flat Text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def tfidf_keyphrases(text, top_k=10):
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 3),
        stop_words="english"
    )
    tfidf_matrix = vectorizer.fit_transform([text])
    scores = tfidf_matrix.toarray()[0]
    terms = vectorizer.get_feature_names_out()

    ranked = sorted(zip(terms, scores), key=lambda x: x[1], reverse=True)
    return [term for term, score in ranked[:top_k]]

In [ ]:
def evaluate_tfidf(dataset, top_k=10):
    total_tp, total_fp, total_fn = 0, 0, 0

    for ex in dataset:
        text = ex["title"] + " " + ex["abstract"]

        if isinstance(ex["keywords"], list):
            gold = set([k.lower() for k in ex["keywords"]])
        else:
            gold = set([k.strip().lower() for k in ex["keywords"].split(";")])

        pred = set(tfidf_keyphrases(text, top_k))

        tp = len(gold & pred)
        fp = len(pred - gold)
        fn = len(gold - pred)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall = total_tp / (total_tp + total_fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return precision, recall, f1

In [ ]:
tfidf_p, tfidf_r, tfidf_f1 = evaluate_tfidf(dataset["validation"])

print("TF-IDF Baseline")
print("Precision:", tfidf_p)
print("Recall:", tfidf_r)
print("F1-score:", tfidf_f1)

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Method": ["TF-IDF (Flat Text)", "SciBERT (Section-Aware)"],
    "Precision": [tfidf_p, precision],
    "Recall": [tfidf_r, recall],
    "F1-score": [tfidf_f1, f1]
})

results

In [ ]:
results_main = pd.DataFrame({
    "Method": [
        "TF-IDF (Flat Text)",

        "SciBERT (Section-Aware)"
    ],
    "Precision": [
        tfidf_p,
        precision
    ],
    "Recall": [
        tfidf_r,
        recall
    ],
    "F1-score": [
        tfidf_f1,
        f1
    ]
})

results_main

# Checking the model working

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATH = "./final_scibert_keyphrase"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

In [ ]:
id2label = {
    0: "O",
    1: "B-KEY",
    2: "I-KEY"
}

In [ ]:
import torch.nn.functional as F

def extract_keyphrases(title, abstract, threshold=0.4):
    text = f"[TITLE] {title} [ABSTRACT] {abstract}"
    words = text.split()

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    word_ids = encoded.word_ids()
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    # Softmax probabilities
    probs = F.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
    preds = np.argmax(probs, axis=-1)

    keyphrases = []
    current_phrase = []
    prev_word_id = None

    for i, word_id in enumerate(word_ids):
        if word_id is None:
            continue

        word = words[word_id]
        key_prob = probs[i][1] + probs[i][2]  # B-KEY + I-KEY

        if word_id != prev_word_id:
            if key_prob >= threshold:
                current_phrase.append(word)
            else:
                if current_phrase:
                    keyphrases.append(" ".join(current_phrase))
                    current_phrase = []

        prev_word_id = word_id

    if current_phrase:
        keyphrases.append(" ".join(current_phrase))

    # Clean & normalize
    cleaned = []
    for kp in keyphrases:
        kp = kp.replace("[TITLE]", "").replace("[ABSTRACT]", "").strip().lower()
        if kp:
            cleaned.append(kp)

    return list(set(cleaned))

In [ ]:
title = "Federated Learning for Privacy Preserving Medical AI"

abstract = """
This proposal introduces a decentralized learning framework
that enables secure model training across hospitals without
sharing sensitive patient data.
"""
print(extract_keyphrases(title, abstract, threshold=0.1))
